In [1]:
import pandas as pd

# încărcare fișier
file_path = "raw_results.csv"
df = pd.read_csv(file_path)

# verificare coloane necesare
required = {'retea', 'sursa', 'p', 'run_id', 'ticks', 'nr_infectati'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Lipsesc coloane în fișierul de intrare: {missing}")

# ajustare indexare DOAR pentru rețelele care pornesc de la 1
# dacă într-o rețea există deja nodul 0, păstrăm indexarea originală
has_zero = df.groupby('retea')['sursa'].transform(lambda x: (x == 0).any())
df.loc[~has_zero, 'sursa'] = df.loc[~has_zero, 'sursa'] + 1

# păstrează doar ultimul rând al fiecărui run
idx = df.groupby(['retea', 'sursa', 'p', 'run_id'])['ticks'].idxmax()

final_per_run = (
    df.loc[idx, ['retea', 'sursa', 'p', 'run_id', 'ticks', 'nr_infectati']]
      .sort_values(['retea', 'p', 'sursa', 'run_id'])
      .rename(columns={
          'retea': 'network',
          'sursa': 'source_vertex',
          'p': 'infection_probability',
          'run_id': 'run',
          'ticks': 'tick',
          'nr_infectati': 'total_infected'
      })
      .reset_index(drop=True)
)

# stochastic centrality = media totalului infectat pe toate rundele
summary = (
    final_per_run
      .groupby(['network', 'infection_probability', 'source_vertex'], as_index=False)
      .agg(
          mean_total_infected=('total_infected', 'mean'),
          median_total_infected=('total_infected', 'median'),
          variance_total_infected=('total_infected', 'var'),
          std_total_infected=('total_infected', 'std'),
          skewness_total_infected=('total_infected', 'skew'),
          min_total_infected=('total_infected', 'min'),
          max_total_infected=('total_infected', 'max'),
          mean_final_tick=('tick', 'mean')
      )
      .sort_values(
          ['network', 'infection_probability', 'mean_total_infected'],
          ascending=[True, True, False]
      )
      .reset_index(drop=True)
)

# salvare rezultate
final_per_run.to_csv("final_per_run.csv", index=False)
summary.to_csv("stochastic_centrality_summary.csv", index=False)

print("All done!")


All done!
